# Notebook 3 : SQL

In [ ]:
# Décommenter la ligne suivante pour installer ibis
#%pip install 'ibis-framework[sqlite]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 17.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.9/670.9 kB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [rich]7/8 [rich]framework]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sqlite3

import pandas as pd
import ibis

from ibis import _

ibis.options.interactive = True

query_tables = "SELECT name FROM sqlite_master WHERE type='table'"

## STAR

Nous considérons les données des stations de vélos en libre service [STAR](https://www.star.fr/) de Rennes Métropole. Une copie de la base SQLite est disponible dans le fichier `star.db`. Nous utilisons d'abord Pandas pour répondre aux questions, puis Ibis.

1. Se connecter à la base de données et afficher la liste des tables à l'aide de la fonction `read_sql` de Pandas et de la requête `query_tables`.

In [6]:
con = sqlite3.connect("data/star.db")
pd.read_sql(query_tables, con)

,name
0,Topologie
1,Etat


2. Récupérer le contenu de la table `Etat` dans un dataframe et afficher la liste des variables disponibles. Même question pour la table `Topologie`.

In [ ]:
#ici attention on charge tt la table en python
etat_df = pd.read_sql("SELECT * FROM Etat", con)
etat_df


,id,nom,latitude,longitude,etat,nb_emplacements,emplacements_disponibles,velos_disponibles,date,data
0,1,République,48.110026,-1.678037,En fonctionnement,30,25,5,1.524646e+09,2018-04-25 08:47:04
1,2,Mairie,48.111624,-1.678757,En fonctionnement,24,6,18,1.524646e+09,2018-04-25 08:47:04
2,3,Champ Jacquet,48.112764,-1.680062,En fonctionnement,24,8,16,1.524646e+09,2018-04-25 08:47:04
3,10,Musée Beaux-Arts,48.109601,-1.674080,En fonctionnement,16,4,12,1.524646e+09,2018-04-25 08:47:04
4,12,TNB,48.107748,-1.673711,En fonctionnement,28,16,12,1.524646e+09,2018-04-25 08:47:04
...,...,...,...,...,...,...,...,...,...,...
78,62,Clemenceau,48.093292,-1.674116,En fonctionnement,22,18,4,1.524646e+09,2018-04-25 08:47:04
79,66,Bréquigny Piscine,48.089621,-1.690242,En fonctionnement,24,19,5,1.524646e+09,2018-04-25 08:47:04
80,69,Champs Manceaux,48.091114,-1.682284,En fonctionnement,18,12,6,1.524646e+09,2018-04-25 08:47:04
81,85,La Courrouze,48.098909,-1.694523,En fonctionnement,20,16,4,1.524646e+09,2018-04-25 08:47:04


In [ ]:
#idem ici cf cellule plus haut
topologie =  pd.read_sql("SELECT * FROM Topologie", con)
topologie

,id,nom,adresse_numero,adresse_voie,commune,latitude,longitude,id_correspondance,mise_en_service,nb_emplacements,id_proche_1,id_proche_2,id_proche_3,terminal_cb
0,1,République,19,Quai Lamartine,Rennes,48.110026,-1.678037,15006.0,14417.0,30,2,11,10,true
1,2,Mairie,12,Galerie du Théâtre,Rennes,48.111624,-1.678757,NaN,14417.0,24,1,3,9,true
2,3,Champ Jacquet,8,Rue du Champ Jacquet,Rennes,48.112764,-1.680062,NaN,14417.0,24,2,5,1,false
3,10,Musée Beaux-Arts,3,Avenue Jean Janvier,Rennes,48.109601,-1.674080,NaN,14417.0,16,12,1,36,false
4,12,TNB,1,Square de Kergus,Rennes,48.107748,-1.673711,NaN,14417.0,28,10,16,11,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,62,Clemenceau,1,Avenue Henri Fréville,Rennes,48.093292,-1.674116,15010.0,14501.0,22,63,61,67,false
79,66,Bréquigny Piscine,10,Boulevard Albert 1er,Rennes,48.089621,-1.690242,NaN,14501.0,24,65,69,70,false
80,69,Champs Manceaux,14,Rue Louis et René Moine,Rennes,48.091114,-1.682284,NaN,14501.0,18,66,62,63,false
81,85,La Courrouze,None,Rue Claude Bernard,Rennes,48.098909,-1.694523,NaN,16527.0,20,20,31,70,false


3. Sélectionner l'identifiant `id`, le nom `nom` et l'identifiant de la station la plus proche `id_proche_1` depuis la table `Topologie`.

In [11]:
topologie[["id", "nom", "id_proche_1"]]


,id,nom,id_proche_1
0,1,République,2
1,2,Mairie,1
2,3,Champ Jacquet,2
3,10,Musée Beaux-Arts,12
4,12,TNB,10
...,...,...,...
78,62,Clemenceau,63
79,66,Bréquigny Piscine,65
80,69,Champs Manceaux,66
81,85,La Courrouze,20


4. Faire une jointure sur la table précédente pour créer une table qui contient la liste des stations avec l'identifiant, le nom et le nom de la station la plus proche associée à l'identifiant `id_proche_1`. Les variables utilisées comme clés sont différents, penser à utiliser les arguments `left_on` et `right_on` de la méthode `merge`.

In [13]:
topologie[["id", "nom", "id_proche_1"]].merge(topologie[["id","nom"]], left_on = "id_proche_1", right_on = "id")

,id_x,nom_x,id_proche_1,id_y,nom_y
0,1,République,2,2,Mairie
1,2,Mairie,1,1,République
2,3,Champ Jacquet,2,2,Mairie
3,10,Musée Beaux-Arts,12,12,TNB
4,12,TNB,10,10,Musée Beaux-Arts
...,...,...,...,...,...
76,55,Préfecture,60,60,Cucillé
77,62,Clemenceau,63,63,Henri Fréville
78,69,Champs Manceaux,66,66,Bréquigny Piscine
79,85,La Courrouze,20,20,Pont de Nantes


5. Ajouter à la table précédente la distance entre la station et la station la plus proche.

In [18]:
df =(topologie[["id","nom","id_proche_1","latitude","longitude"]].merge(topologie[["id","nom","id_proche_1","latitude","longitude"]],left_on = "id_proche_1", right_on = "id"))

df["distance"] = (df["latitude_x"]-df["latitude_y"])**2

df = (df.drop(columns=["id","nom","id_proche_1","latitude","longitude"]).rename(columns = {"id_x","id","nom_x","noms_y","nom_proche"})
)

df

KeyError: "['id', 'nom', 'id_proche_1', 'latitude', 'longitude'] not found in axis"

6. Créer une table avec le nom des trois stations les plus proches du point GPS *(48.1179151,-1.7028661)* classées par ordre de distance et le nombre de vélos disponibles dans ces stations.

7. Reprendre les questions précédentes en utilisant le module `ibis`. Pour les jointures, utiliser la méthode `left_join`.

In [22]:
con = ibis.sqlite.connect("data/star.db")

In [24]:
con.tables

Tables
------
- Etat
- Topologie

In [26]:
etat = con.table("Etat")
topologieibis = con.table("Topologie")

In [27]:
etat

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ latitude  ┃ longitude ┃ etat              ┃ nb_emplacements ┃ emplacements_disponibles ┃ velos_disponibles ┃ date         ┃ data                ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ int64 │ string             │ float64   │ float64   │ string            │ int64           │ int64                    │ int64             │ float64      │ string              │
├───────┼────────────────────┼───────────┼───────────┼───────────────────┼─────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────────┤
│     1 │ République         │ 48.110026 │ -1.678037 │ En fonctionnement │              30 │                       25 │                 5 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│     2 │ Mairie             │ 48.111624 │ -1.678757 │ En fonctionnement │              24 │                        6 │                18 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│     3 │ Champ Jacquet      │ 48.112764 │ -1.680062 │ En fonctionnement │              24 │                        8 │                16 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    10 │ Musée Beaux-Arts   │ 48.109601 │ -1.674080 │ En fonctionnement │              16 │                        4 │                12 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    12 │ TNB                │ 48.107748 │ -1.673711 │ En fonctionnement │              28 │                       16 │                12 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    14 │ Laënnec            │ 48.106847 │ -1.665817 │ En fonctionnement │              16 │                        3 │                13 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    17 │ Charles de Gaulle  │ 48.105111 │ -1.677119 │ En fonctionnement │              24 │                       17 │                 7 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    20 │ Pont de Nantes     │ 48.102015 │ -1.684015 │ En fonctionnement │              20 │                        9 │                11 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    22 │ Oberthur           │ 48.113550 │ -1.661858 │ En fonctionnement │              20 │                       13 │                 7 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│    25 │ Office de Tourisme │ 48.110294 │ -1.683106 │ En fonctionnement │              10 │                        6 │                 4 │ 1.524646e+09 │ 2018-04-25 08:47:04 │
│     … │ …                  │         … │         … │ …                 │               … │                        … │                 … │            … │ …                   │
└───────┴────────────────────┴───────────┴───────────┴───────────────────┴─────────────────┴──────────────────────────┴───────────────────┴──────────────┴─────────────────────┘

In [28]:
topologieibis

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ adresse_numero ┃ adresse_voie                               ┃ commune ┃ latitude  ┃ longitude ┃ id_correspondance ┃ mise_en_service ┃ nb_emplacements ┃ id_proche_1 ┃ id_proche_2 ┃ id_proche_3 ┃ terminal_cb ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int64 │ string             │ string         │ string                                     │ string  │ float64   │ float64   │ int64             │ float64         │ int64           │ int64       │ int64       │ int64       │ string      │
├───────┼────────────────────┼────────────────┼────────────────────────────────────────────┼─────────┼───────────┼───────────┼───────────────────┼─────────────────┼─────────────────┼─────────────┼─────────────┼─────────────┼─────────────┤
│     1 │ République         │ 19             │ Quai Lamartine                             │ Rennes  │ 48.110026 │ -1.678037 │             15006 │         14417.0 │              30 │           2 │          11 │          10 │ true        │
│     2 │ Mairie             │ 12             │ Galerie du Théâtre                         │ Rennes  │ 48.111624 │ -1.678757 │              NULL │         14417.0 │              24 │           1 │           3 │           9 │ true        │
│     3 │ Champ Jacquet      │ 8              │ Rue du Champ Jacquet                       │ Rennes  │ 48.112764 │ -1.680062 │              NULL │         14417.0 │              24 │           2 │           5 │           1 │ false       │
│    10 │ Musée Beaux-Arts   │ 3              │ Avenue Jean Janvier                        │ Rennes  │ 48.109601 │ -1.674080 │              NULL │         14417.0 │              16 │          12 │           1 │          36 │ false       │
│    12 │ TNB                │ 1              │ Square de Kergus                           │ Rennes  │ 48.107748 │ -1.673711 │              NULL │         14417.0 │              28 │          10 │          16 │          11 │ false       │
│    14 │ Laënnec            │ 11             │ Boulevard René Théophile Hyacinthe Laënnec │ Rennes  │ 48.106847 │ -1.665817 │              NULL │         14417.0 │              16 │          35 │          15 │          12 │ false       │
│    17 │ Charles de Gaulle  │ 2              │ Cours des Alliés                           │ Rennes  │ 48.105111 │ -1.677119 │             15007 │         14417.0 │              24 │          16 │          18 │          11 │ true        │
│    20 │ Pont de Nantes     │ 32             │ Rue du Docteur Francis Joly                │ Rennes  │ 48.102015 │ -1.684015 │              NULL │         14417.0 │              20 │          43 │          18 │          70 │ false       │
│    22 │ Oberthur           │ 4              │ Rue Gutenberg                              │ Rennes  │ 48.113550 │ -1.661858 │              NULL │         14417.0 │              20 │          44 │          35 │          42 │ false       │
│    25 │ Office de Tourisme │ 2              │ Rue Le Bouteiller                          │ Rennes  │ 48.110294 │ -1.683106 │              NULL │         14417.0 │              10 │          24 │           1 │           2 │ true        │
│     … │ …                  │ …              │ …                                          │ …       │         … │         … │                 … │               … │               … │           … │           … │           … │ …           │
└───────┴────────────────────┴────────────────┴────────────────────────────────────────────┴─────────┴───────────┴───────────┴───────────────────┴─────────────────┴────────────

In [29]:
topologieibis.select(["id", "nom", "id_proche_1"])

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ id_proche_1 ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int64 │ string             │ int64       │
├───────┼────────────────────┼─────────────┤
│     1 │ République         │           2 │
│     2 │ Mairie             │           1 │
│     3 │ Champ Jacquet      │           2 │
│    10 │ Musée Beaux-Arts   │          12 │
│    12 │ TNB                │          10 │
│    14 │ Laënnec            │          35 │
│    17 │ Charles de Gaulle  │          16 │
│    20 │ Pont de Nantes     │          43 │
│    22 │ Oberthur           │          44 │
│    25 │ Office de Tourisme │          24 │
│     … │ …                  │           … │
└───────┴────────────────────┴─────────────┘

In [32]:
tbl = topologieibis.select(["id", "nom", "id_proche_1"])

(
    tbl
    .left_join(topologieibis, tbl.id_proche_1 == topologieibis.id)
    .select(tbl.id, tbl.nom, nom_proche = topologieibis.nom)



)

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ nom_proche         ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ int64 │ string             │ string             │
├───────┼────────────────────┼────────────────────┤
│     1 │ République         │ Mairie             │
│     2 │ Mairie             │ République         │
│     3 │ Champ Jacquet      │ Mairie             │
│    10 │ Musée Beaux-Arts   │ TNB                │
│    12 │ TNB                │ Musée Beaux-Arts   │
│    14 │ Laënnec            │ Pont de Châteaudun │
│    17 │ Charles de Gaulle  │ Champs Libres      │
│    20 │ Pont de Nantes     │ Cité Judiciaire    │
│    22 │ Oberthur           │ Metz - Sévigné     │
│    25 │ Office de Tourisme │ Place de Bretagne  │
│     … │ …                  │ …                  │
└───────┴────────────────────┴────────────────────┘

8. (*Bonus*) Écrire des requêtes SQL pour obtenir les résultats demandés dans les questions 3 à 6. La fonction `to_sql` pourra être utilisée pour de l'aide.

## Musique

Le dépôt GitHub [lerocha/chinook-database](https://github.com/lerocha/chinook-database) met à disposition des bases de données de bibliothèques musicales. Une copie de la base SQLite est disponible dans le fichier `chinook.db`.

1. Utiliser le module `ibis` pour vous connecter à la base de données et explorer les tables formant le jeu de données pour le découvrir. En particulier, remarquer comment les tables `Playlist`, `PlaylistTrack` et `Track` sont liées entre elles.

2. Quelles sont les playlists qui contiennent le plus de pistes ?

3. Construire une table contenant les informations suivantes sur la playlist `Classical` : le titre de chaque piste ainsi que le titre de l'album dont cette piste est tirée.

4. (*Bonus*) Écrire une requête SQL donnant le résultat de la question précédente. La fonction `to_sql` pourra être utilisée pour de l'aide.